In [1]:
# IMPORTS
from datetime import datetime, timezone
from io import StringIO

import certifi
import numpy as np
import pandas as pd
import requests

#Fin Data Sources
import yfinance as yf

Question 1: [Index] S&P 500 Stocks Added to the Index. Which year had the highest number of additions (starting from 2020)?
     



In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, verify=certifi.where(), timeout=30)
response.raise_for_status()  # good practice — catches HTTP errors early

tables = pd.read_html(StringIO(response.text))
sp500_df = tables[0]

In [3]:
sp500_df['Date added'] = pd.to_datetime(sp500_df['Date added'], errors='coerce')
sp500_df['Year Added'] = sp500_df['Date added'].dt.year
additions_per_year = sp500_df.groupby('Year Added')['Security'].count()

In [4]:
# Find the maximum count after 2020
additions_after_2020 = additions_per_year[additions_per_year.index > 2020]
max_additions_count = additions_after_2020.max()

# Get ALL years that match the maximum count
max_years_mask = additions_after_2020 == max_additions_count
max_years = additions_after_2020[max_years_mask].index.tolist()

# Print results handling single vs. multiple years
years_str = ", ".join(map(str, max_years))
print(
    f"The year(s) with the most additions after 2020: {years_str} "
    f"(with {max_additions_count} additions)."
)

The year(s) with the most additions after 2020: 2025 (with 18 additions).


In [5]:
# Additional: How many current S&P 500 stocks have been in the index for more than
# 20 years?
current_year = datetime.now(timezone.utc).year
sp500_df['Years in Index'] = current_year - sp500_df['Year Added']
long_term_stocks_count = (sp500_df['Years in Index'] > 20).sum()
print(
    "Number of current S&P 500 stocks that have been in the index for more than "
    f"20 years: {long_term_stocks_count}."
)

Number of current S&P 500 stocks that have been in the index for more than 20 years: 218.


Question 2. [Macro] Indexes YTD (as of 21 August 2026).\
How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

In [6]:
start_date='2026-01-01'
end_date='2026-08-21'

In [7]:
SP500_LABEL = "United States - S&P 500"

indices_tickers = {
    SP500_LABEL: "^GSPC",
    "China - Shanghai Composite": "000001.SS",
    "Hong Kong - HANG SENG INDEX": "^HSI",
    "Australia - S&P/ASX 200": "^AXJO",
    "India - Nifty 50": "^NSEI",
    "Canada - S&P/TSX Composite": "^GSPTSE",
    "Germany - DAX": "^GDAXI",
    "United Kingdom - FTSE 100": "^FTSE",
    "Japan - Nikkei 225": "^N225",
    "Mexico - IPC Mexico": "^MXX",
    "Brazil - Ibovespa": "^BVSP"
}

In [8]:
def download_price_history(ticker, start, end):
    # Single-level index (multi_level_index=False) is easier to work with downstream.
    try:
        return yf.download(ticker, start=start, end=end, multi_level_index=False)
    except Exception as e:
        print(f"Error occurred while downloading data for {ticker}: {e}")
        return None

In [9]:
all_indices_data = {}
for index_name, ticker in indices_tickers.items():
    data = download_price_history(ticker, start_date, end_date)
    if data is not None:
        all_indices_data[index_name] = data
        print(f"Downloaded data for {index_name} ({ticker})")


[*********************100%***********************]  1 of 1 completed



[*********************100%***********************]  1 of 1 completed



[*********************100%***********************]  1 of 1 completed

Downloaded data for United States - S&P 500 (^GSPC)
Downloaded data for China - Shanghai Composite (000001.SS)
Downloaded data for Hong Kong - HANG SENG INDEX (^HSI)



[*********************100%***********************]  1 of 1 completed

...


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed

Downloaded data for Japan - Nikkei 225 (^N225)
Downloaded data for Mexico - IPC Mexico (^MXX)



[*********************100%***********************]  1 of 1 completed

Downloaded data for Brazil - Ibovespa (^BVSP)


In [10]:
print(all_indices_data["Germany - DAX"].head())

                   Close          High           Low          Open    Volume
Date                                                                        
2026-01-02  24539.339844  24676.789062  24448.980469  24499.509766  48788700
2026-01-05  24868.689453  24872.919922  24639.679688  24681.349609  53518200
2026-01-06  24892.199219  24969.169922  24836.330078  24904.679688  57420800
2026-01-07  25122.259766  25122.460938  24982.619141  24995.000000  57897000
2026-01-08  25127.460938  25217.519531  25024.650391  25128.460938  57587500


In [11]:
# Year to date returns calculation
ytd_returns = {}
for index_name, data in all_indices_data.items():
    start_price = data['Close'].iloc[0]
    end_price = data['Close'].iloc[-1]
    ytd_return = (end_price - start_price) / start_price * 100
    ytd_returns[index_name] = ytd_return

In [12]:
ytd_returns_df = pd.DataFrame(
    list(ytd_returns.items()), columns=['Index', 'YTD Return']
)
print(ytd_returns_df)

                          Index  YTD Return
0       United States - S&P 500   11.412019
1    China - Shanghai Composite   -2.974985
2   Hong Kong - HANG SENG INDEX   -2.429832
3       Australia - S&P/ASX 200    4.078920
4              India - Nifty 50   -7.322959
5    Canada - S&P/TSX Composite   14.057466
6                 Germany - DAX    5.883203
7     United Kingdom - FTSE 100    8.010176
8            Japan - Nikkei 225   27.750745
9           Mexico - IPC Mexico    0.324972
10            Brazil - Ibovespa    4.601997


In [13]:
# Now we can compare the YTD returns of all indexes with the S&P 500 and count
# how many have better returns.
sp500_ytd_return = ytd_returns[SP500_LABEL]
performance_comparison = {
    index: ret for index, ret in ytd_returns.items() if ret > sp500_ytd_return
}
better_performance_count = len(performance_comparison)
print(
    "Number of indexes with better YTD returns than the S&P 500 as of "
    f"August 21, 2026: {better_performance_count}."
)

Number of indexes with better YTD returns than the S&P 500 as of August 21, 2026: 2.


Additional: How many of these indexes have better returns than the S&P 500 over 3, 5, and 10 year periods? \
 Do you see the same trend? \
 Note: For simplicity, ignore currency conversion effects.

In [14]:
start_date_3y = '2023-08-21'
start_date_5y = '2021-08-21'
start_date_10y = '2016-08-21'

In [15]:
# We will download the data for 10 years for all indices and then calculate the
# returns for 3, 5, and 10 year periods.
all_indices_data_10y = {}
for index_name, ticker in indices_tickers.items():
    data = download_price_history(ticker, start_date_10y, end_date)
    if data is not None:
        all_indices_data_10y[index_name] = data
        print(f"Downloaded 10-year data for {index_name} ({ticker})")


[*********************100%***********************]  1 of 1 completed

Downloaded 10-year data for United States - S&P 500 (^GSPC)




[*********************100%***********************]  1 of 1 completed

Downloaded 10-year data for China - Shanghai Composite (000001.SS)



[*********************100%***********************]  1 of 1 completed

...

Downloaded 10-year data for United Kingdom - FTSE 100 (^FTSE)




[*********************100%***********************]  1 of 1 completed

Downloaded 10-year data for Japan - Nikkei 225 (^N225)
Downloaded 10-year data for Mexico - IPC Mexico (^MXX)



[*********************100%***********************]  1 of 1 completed

Downloaded 10-year data for Brazil - Ibovespa (^BVSP)


In [16]:
# Write a function to calculate the returns for any given period and compare with
# the S&P 500, so it can be reused for the 3, 5, and 10 year periods.
def calculate_returns_and_compare(start_date, end_date, all_indices_data):
    returns = {}
    for index_name, data in all_indices_data.items():
        # Filter data for the given period
        data_period = data[data.index >= start_date]
        if not data_period.empty:
            start_price = data_period['Close'].iloc[0]
            end_price = data_period['Close'].iloc[-1]
            return_value = (end_price - start_price) / start_price * 100
            returns[index_name] = return_value
    # Print the returns for the given period and sort them in descending order
    sorted_returns = dict(
        sorted(returns.items(), key=lambda item: item[1], reverse=True)
    )
    returns_df = pd.DataFrame(list(sorted_returns.items()), columns=['Index', 'Return'])
    print(f"Returns from {start_date} to {end_date}:")
    print(returns_df.to_string(index=False))
    print("*" * 80)
    # Compare with S&P 500
    sp500_return = returns[SP500_LABEL]
    performance_comparison = {
        index: ret for index, ret in returns.items() if ret > sp500_return
    }
    better_performance_count = len(performance_comparison)
    return better_performance_count

In [17]:
# Now apply this function for 3, 5, and 10 year periods.
periods = {
    "3-year": start_date_3y,
    "5-year": start_date_5y,
    "10-year": start_date_10y,
}
for label, period_start in periods.items():
    count = calculate_returns_and_compare(period_start, end_date, all_indices_data_10y)
    print(
        f"Number of indexes with better {label} returns than the S&P 500 "
        f"as of August 21, 2026: {count}."
    )
    print("\n")

Returns from 2023-08-21 to 2026-08-21:
                      Index     Return
         Japan - Nikkei 225 109.774894
 Canada - S&P/TSX Composite  83.803798
    United States - S&P 500  73.671808
              Germany - DAX  66.522927
  United Kingdom - FTSE 100  48.091715
          Brazil - Ibovespa  46.752135
Hong Kong - HANG SENG INDEX  45.821192
    Australia - S&P/ASX 200  27.662143
 China - Shanghai Composite  26.212293
           India - Nifty 50  24.947664
        Mexico - IPC Mexico  21.163791
********************************************************************************
Number of indexes with better 3-year returns than the S&P 500 as of August 21, 2026: 2.


Returns from 2021-08-21 to 2026-08-21:
                      Index     Return
         Japan - Nikkei 225 140.838767
 Canada - S&P/TSX Composite  77.588828
    United States - S&P 500  70.579514
              Germany - DAX  63.901995
  United Kingdom - FTSE 100  51.191450
           India - Nifty 50  46.891305
          

Question 3. [Index] S&P 500 Market Corrections Analysis \
Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

For this task, define a correction as an event when a stock index goes down by at least 5% from the most recent all-time high.

Steps:

Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)\
Identify all-time high closing price points (where price exceeds all previous days' prices)\
For each pair of consecutive all-time highs, find the minimum price in between\
Calculate drawdown percentages: (high - low) / high × 100\
Filter for corrections with at least 5% drawdown\
Calculate the duration in days for each correction period\
Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

In [18]:
start_date = '1950-01-01'
end_date = datetime.now(timezone.utc).strftime('%Y-%m-%d')

In [19]:
sp500_data = yf.download(
    "^GSPC", start=start_date, end=end_date, multi_level_index=False
)


[*********************100%***********************]  1 of 1 completed

In [20]:
# Calculate all-time highs for the S&P 500 for each day, stored as 'All_Time_High'.
sp500_data['All_Time_High'] = sp500_data['Close'].cummax()

# Calculate drawdowns: (All_Time_High - Close) / All_Time_High * 100
sp500_data['Drawdown'] = (
    (sp500_data['All_Time_High'] - sp500_data['Close'])
    / sp500_data['All_Time_High']
    * 100
)

In [21]:
correction_threshold = 5
peak_corrections = []
is_new_ath = sp500_data['Close'] == sp500_data['All_Time_High']
new_ath_dates = sp500_data[is_new_ath].index.tolist()

In [22]:
# Iterate through the ATH dates to find the subsequent troughs and check if the
# drawdown exceeds the correction threshold. Report the start date, end date,
# drawdown percentage, and duration (in days) of each correction.
for i in range(len(new_ath_dates) - 1):
    period_start = new_ath_dates[i]
    period_end = new_ath_dates[i + 1]

    # Get the data between the two ATH dates
    period_data = sp500_data.loc[period_start:period_end]

    # Find the minimum close price in this period (the trough)
    trough_price = period_data['Close'].min()
    trough_date = period_data['Close'].idxmin()

    # Calculate the drawdown percentage from the ATH to the trough
    ath_price = sp500_data.loc[period_start, 'All_Time_High']
    drawdown_percentage = (ath_price - trough_price) / ath_price * 100

    # Check if the drawdown exceeds the correction threshold
    if drawdown_percentage >= correction_threshold:
        duration_days = (trough_date - period_start).days
        peak_corrections.append({
            'Start Date': period_start,
            'End Date': trough_date,
            'Drawdown Percentage': drawdown_percentage,
            'Duration (days)': duration_days
        })

In [23]:
# Find the 10 largest corrections by drawdown percentage
largest_corrections = sorted(
    peak_corrections, key=lambda x: x['Drawdown Percentage'], reverse=True
)[:10]
print("The 10 largest corrections in the S&P 500 since 1950 are:")
for correction in largest_corrections:
    print(
        f"Start Date: {correction['Start Date'].date()}, "
        f"End Date: {correction['End Date'].date()}, "
        f"Drawdown Percentage: {correction['Drawdown Percentage']:.2f}%, "
        f"Duration: {correction['Duration (days)']} days"
    )

The 10 largest corrections in the S&P 500 since 1950 are:
Start Date: 2007-10-09, End Date: 2009-03-09, Drawdown Percentage: 56.78%, Duration: 517 days
Start Date: 2000-03-24, End Date: 2002-10-09, Drawdown Percentage: 49.15%, Duration: 929 days
Start Date: 1973-01-11, End Date: 1974-10-03, Drawdown Percentage: 48.20%, Duration: 630 days
Start Date: 1968-11-29, End Date: 1970-05-26, Drawdown Percentage: 36.06%, Duration: 543 days
Start Date: 2020-02-19, End Date: 2020-03-23, Drawdown Percentage: 33.92%, Duration: 33 days
Start Date: 1987-08-25, End Date: 1987-12-04, Drawdown Percentage: 33.51%, Duration: 101 days
Start Date: 1961-12-12, End Date: 1962-06-26, Drawdown Percentage: 27.97%, Duration: 196 days
Start Date: 1980-11-28, End Date: 1982-08-12, Drawdown Percentage: 27.11%, Duration: 622 days
Start Date: 2022-01-03, End Date: 2022-10-12, Drawdown Percentage: 25.43%, Duration: 282 days
Start Date: 1966-02-09, End Date: 1966-10-07, Drawdown Percentage: 22.18%, Duration: 240 days


In [24]:
# 25th, 50th (median), and 75th percentiles for correction durations and drawdowns
durations = [c['Duration (days)'] for c in peak_corrections]
drawdowns = [c['Drawdown Percentage'] for c in peak_corrections]

dur_25, dur_50, dur_75 = np.percentile(durations, [25, 50, 75])
dd_25, dd_50, dd_75 = np.percentile(drawdowns, [25, 50, 75])

print(f"Total corrections (>= {correction_threshold}%): {len(peak_corrections)}")
print(
    f"Duration (days)  -> 25th: {round(dur_25)}, Median: {round(dur_50)}, "
    f"75th: {round(dur_75)}"
)
print(f"Drawdown (%)     -> 25th: {dd_25:.2f}, Median: {dd_50:.2f}, 75th: {dd_75:.2f}")

Total corrections (>= 5%): 74
Duration (days)  -> 25th: 22, Median: 40, 75th: 86
Drawdown (%)     -> 25th: 6.23, Median: 7.99, 75th: 14.02


Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN) \
Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

In [25]:
ticker = 'AMZN'
ticker_data = yf.Ticker(ticker)
amazon_earnings = ticker_data.get_earnings_dates(limit=25)
amazon_price_data = ticker_data.history(period="max")  # All the historical price data

In [26]:
# 2-day return across 3 consecutive trading days (Day 1, Day 2, Day 3): computed as
# Close_Day3 / Close_Day1 - 1. Day 2 may correspond to the earnings announcement.
amazon_price_data['2_day_return'] = (
    amazon_price_data['Close'].shift(-1) / amazon_price_data['Close'].shift(1) - 1
)

In [27]:
# Normalize both indices to date-only, tz-naive, so they actually align on merge
amazon_earnings.index = (
    pd.to_datetime(amazon_earnings.index).tz_localize(None).normalize()
)
amazon_price_data.index = (
    pd.to_datetime(amazon_price_data.index).tz_localize(None).normalize()
)

In [28]:
# Filter for positive earnings surprises, then merge with 2-day returns to
# calculate the median 2-day return and its correlation with surprise magnitude.
positive_surprises = amazon_earnings[amazon_earnings['Surprise(%)'] > 0]
merged_data = positive_surprises.merge(
    amazon_price_data[['2_day_return']],
    left_index=True,
    right_index=True,
    how='left',
)
print(f"Merged data shape: {merged_data.shape}")
merged_data = merged_data.dropna(subset=['2_day_return', 'Surprise(%)'])
print(f"Merged data shape after dropping NaNs: {merged_data.shape}")

Merged data shape: (20, 4)
Merged data shape after dropping NaNs: (20, 4)


In [29]:
median_2d_return = merged_data['2_day_return'].median()
correlation = merged_data['2_day_return'].corr(merged_data['Surprise(%)'])

print(f"Positive surprise events: {len(merged_data)}")
print(f"Median 2-day return: {median_2d_return:.4%}")
print(f"Correlation (2-day return vs Surprise %): {correlation:.4f}")

Positive surprise events: 20
Median 2-day return: 0.3528%
Correlation (2-day return vs Surprise %): 0.3306


Question 5. [Exploratory, optional] Brainstorm potential idea for your capstone project

My professional background includes trading soft commodity markets, dating back approximately 15 years to the early stages of my career. Building on that foundation, I am looking to develop a systematic trading strategy operating on a multi-week to longer-term horizon, designed to capture directional price movements in Raw Sugar No. 11 futures. Given the structural drivers of this market, the framework would need to integrate several key fundamental inputs: production-side news flow (crop conditions, harvest reports, and mill output data, particularly from Brazil and India), supply-demand balance dynamics, currency effects — principally BRL and INR, given their influence on export competitiveness and grower incentives — and crude oil price movements, owing to the well-established linkage between sugar and ethanol markets in Brazil's flex-fuel economy.

Question 6. [Exploratory, optional] Investigate new metrics

UNICA (the Brazilian sugarcane industry association) publishes fortnightly crop production reports covering cane crushing volumes, sugar/ethanol production mix, and regional output for Brazil's Center-South region — the dominant global sugar-producing area. Since Brazilian Center-South supply is a key driver of Raw Sugar No. 11 pricing, this series would serve as a leading fundamental indicator for anticipating shifts in global supply.The reports are only available as PDFs on UNICA's website, with no structured API, so retrieval would require scraping the report archive, downloading the PDFs, and parsing the embedded tables into a clean, time-indexed dataset for analysis.